Modeling Features: One By One
=============================

Multi-wavelength analysis does not necessarily require us to fit all datasets simultaneously. Instead, we can fit one
dataset first in order to infer a robust galaxy model, and then fit the next dataset, using the inferred
model as the starting point.

There are many occasions where this approach is beneficial, for example:

- When certain datasets are worse quality (e.g. lower resolution) than others. Fitting them simultaneously may mean this
  dataset's lower quality makes the model fit less robust. By fitting them one by one, using the inferred model of the
  best dataset first, we can ensure the model-fit is as robust as possible and interpret the results of the lower
  quality datasets more clearly.

- It can often produce faster run times, as although more non-linear searches are performed, each search is faster
  than a search which fits all datasets simultaneously.

- To investigate whether modeling results inferred when we model all datasets simultanoeusly are robust. If the
  result disappears for fits to individual datasets, this may suggest the result is not robust.

To perform modeling one-by-one, we have to make decision about how simple or complex we make the model after
fitting the highest quality dataset. For example, we may:

- Fix the majority of galaxy structural parameters, allowing only the `intensity` values to vary.

- Allow all parameters to vary, but use the highest quality dataset's inferred model as the starting point.

- Whether to account for offsets between the datasets, or to assume the datasets are aligned.

We illustrate different examples in this script, with the appropriate choice depending on your specific science case.

__Model__

This script fits multiple multi-wavelength `Imaging` datasets of a galaxy with a model where:

 - The galaxy's light is a linear parametric `Sersic` bulge and `Exponential` disk.

Two images are fitted, corresponding to a greener ('g' band) and redder image (`r` band).

__Contents__

- **Model:** Description of the galaxy model fitted in this example.
- **Colors:** Defining the multi-wavelength color bands.
- **Pixel Scales:** Setting per-wavelength pixel scales.
- **Dataset:** Loading and plotting each multi-wavelength dataset.
- **Dataset Auto-Simulation:** Automatically simulating data if it does not exist.
- **Mask:** Applying a circular mask to each dataset.
- **Analysis:** Creating analysis objects for each dataset with JAX acceleration.
- **Model:** Composing a Sersic bulge and Exponential disk model with linear light profiles.
- **Search:** Configuring the Nautilus nested sampling non-linear search.
- **Model-Fit:** Fitting the first (highest quality) dataset.
- **Result:** Inspecting the result of the first dataset fit.
- **Second Dataset Bulge Model Fixed:** Fitting the second dataset with the bulge fixed from the first result.
- **Second Dataset Offset:** Fitting for the offset between datasets with all model parameters fixed.

__Start Here Notebook__

If any code in this script is unclear, refer to the `multi_dataset/modeling.ipynb` notebook.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autogalaxy_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autogalaxy as ag
import autogalaxy.plot as aplt

__Colors__

The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).

The strings are used for load each dataset.

In [ ]:
waveband_list = ["g", "r"]

__Pixel Scales__

Every multi-wavelength dataset can have its own unique pixel-scale.

In [ ]:
pixel_scales_list = [0.08, 0.12]

__Dataset__

Load and plot each multi-wavelength dataset, using a list of their waveband colors.

The plotted images show that the datasets have a small offset between them, half a pixel based on the resolution of
the second image.

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "imaging"
dataset_name = "simple"

dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if ag.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/multi_dataset/simulator.py"],
        check=True,
    )


dataset_list = [
    ag.Imaging.from_fits(
        data_path=Path(dataset_path) / f"{waveband}_data.fits",
        psf_path=Path(dataset_path) / f"{waveband}_psf.fits",
        noise_map_path=Path(dataset_path) / f"{waveband}_noise_map.fits",
        pixel_scales=pixel_scales,
    )
    for waveband, pixel_scales in zip(waveband_list, pixel_scales_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

Define a 3.0" circular mask, which includes the emission of the galaxy.

For multi-wavelength modeling, we use the same mask for every dataset whenever possible. This is not
absolutely necessary, but provides a more reliable analysis.

In [ ]:
mask_list = [
    ag.Mask2D.circular(
        shape_native=dataset.shape_native, pixel_scales=dataset.pixel_scales, radius=3.0
    )
    for dataset in dataset_list
]

dataset_list = [
    dataset.apply_mask(mask=mask) for imaging, mask in zip(dataset_list, mask_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Analysis__

We create an `Analysis` object for every dataset.

We do not combine the analyses using a factor graph, like we do in most other example scripts, as we are going to fit 
each dataset one-by-one.

In [ ]:
analysis_list = [
    ag.AnalysisImaging(dataset=dataset, use_jax=True) for dataset in dataset_list
]

__Model__

We compose our galaxy model using `Model` objects, which represent the galaxies we fit to our data. In this 
example we fit a galaxy model where:

 - The galaxy's bulge is a linear parametric `Sersic` bulge [6 parameters]. 
 
 - The galaxy's disk is a linear parametric `Exponential` disk [6 parameters].
 
The number of free parameters and therefore the dimensionality of non-linear parameter space is N=15.

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)
disk = af.Model(ag.lp_linear.Exponential)

galaxy = af.Model(ag.Galaxy, redshift=0.5, bulge=bulge, disk=disk)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

__Search__

In [ ]:
search = af.Nautilus(
    path_prefix=Path("multi_dataset") / "features",
    name="one_by_one__main_dataset",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Model-Fit__

In [ ]:
result = search.fit(model=model, analysis=analysis_list[0])

__Result__

The result object returned by this model-fit is a `Result` object. It is not a list like other examples, because we 
did not use a combined analysis.

In [ ]:
print(result.max_log_likelihood_instance)

Plotting the result's galaxy shows it at this wavelength.

In [ ]:
aplt.subplot_galaxies(galaxies=result.max_log_likelihood_galaxies, grid=result.grids.lp)
aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

__Second Dataset Bulge Model Fixed__

We now fit the second dataset using the inferred model of the first dataset as the starting point.

We compose a simple model where the bulge is fixed to the result of the first dataset fit, and galaxy's disk parameters
are varied. 

This is a bit of a strange model to fit, and is done for illustrative purposes. 

The code below uses the search chaining API to link the priors between model parameters, if you are not
familiar with this feature, checkout the `guides/modeling/chaining` package.

In [ ]:
model = af.Collection(
    galaxies=af.Collection(
        galaxy=af.Model(
            ag.Galaxy,
            redshift=result.instance.galaxies.galaxy.redshift,
            bulge=result.instance.galaxies.galaxy.bulge,
            disk=result.model.galaxies.galaxy.disk,
        )
    ),
)

print(model.info)

search = af.Nautilus(
    path_prefix=Path("multi_dataset") / "features",
    name="one_by_one__second_bulge_fixed",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

result_bulge_fixed = search.fit(model=model, analysis=analysis_list[0])

__Second Dataset Offset__

Multi-wavelength datasets often have offsets between their images, which are due to the different telescope pointings
during the observations.

These offsets are often accounted for during the data reduction process, but may not be perfectly corrected and
have uncertainties associated with them.

Fitting datasets one-by-one offers a straightforward method to account for these offsets, by allowing the offset
between the datasets to vary during the model-fit as two free parameters (y and x).

We now fit for the offset between datasets, keeping all model parameters fixed to the result of the first dataset
fit. 

In this example, the two datasets are not offset, so the model-fit will infer an offset consistent with (0.0", 0.0").

In [ ]:
dataset_model = af.Model(ag.DatasetModel)

dataset_model.grid_offset.grid_offset_0 = af.UniformPrior(
    lower_limit=-0.1, upper_limit=0.1
)
dataset_model.grid_offset.grid_offset_1 = af.UniformPrior(
    lower_limit=-0.1, upper_limit=0.1
)

model = af.Collection(
    dataset_model=dataset_model,
    galaxies=result.instance.galaxies,
)

print(model.info)

search = af.Nautilus(
    path_prefix=Path("multi_dataset") / "features",
    name="one_by_one__dataset_offset",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

result = search.fit(model=model, analysis=analysis_list[0])

Checkout `autogalaxy_workspace/*/results` for a full description of analysing results in **Pyautogalaxy**.